In [29]:
import os, re, json
import pandas as pd
from openai import OpenAI

In [28]:
%%time

INPUT_PATH = "PATH"
OUTPUT_DIR = "PATH"
MODEL = "gpt-5"
BATCH_SIZE = 25            # smaller batches to reduce truncation risk?
os.environ["OPENAI_API_KEY"] = "KEY"

client = OpenAI()  # expects OPENAI_API_KEY in environment

# Read input text
text = open(INPUT_PATH, "r", encoding="utf-8", errors="ignore").read()

# Sentence splitter
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

# Output path
stem = os.path.splitext(os.path.basename(INPUT_PATH))[0]
output_jsonl_path = os.path.join(OUTPUT_DIR, f"{stem}_metaphor_analysis.jsonl")

# Instructions
PROMPT = """Search through the provided text file. For each sentence: 
1. Find metaphor-related words by examining the text on a word-by-word basis. 
2. When a word is used metaphorically or metonymically, identify the metaphor that is used, based on George Lakoff and Mark Johnson's theory of conceptual metaphors. 
3. List the sentence, followed by the word that has been used metaphorically, followed by the target domain of the metaphor and the source domain of the metaphor. 
4. When a word is a new-formation coined, examine the distinct words that are its independent parts according to steps 2 and 3. 
5. If a sentence contains no metaphorically used words, output the sentence as-is. 

For example: 
If the sentence is "Your claims are indefensible", the word "indefensible" is used metaphorically, and the conceptual metaphor used is ARGUMENT IS WAR. 
The response in this case must be: 'Your claims are indefensible.',['indefensible'],ARGUMENT,WAR 
If the sentence is "We need boots on the ground", the word "boots" is used metonymically, and the conceptual metaphor used is PART FOR WHOLE. 
The response in this case must be: 'We need boots on the ground.',['boots'],PART,WHOLE 
"""

GUIDANCE = """Return one JSON object per input sentence (JSON Lines). Fields:
- sentence: the original sentence (string, exactly as given, keep punctuation)
- words: array of strings for metaphorical/metonymic words (empty if none)
- target_domain: string (or empty)
- source_domain: string (or empty)

No extra text, no headers, no explanations—only JSON objects, one per line, in the same order as the input sentences.
"""

# Write once, append each lines from each batch
with open(output_jsonl_path, "w", encoding="utf-8") as f:
    for start in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[start:start+BATCH_SIZE]
        if not batch:
            continue

        numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(batch))
        user_input = (
            PROMPT + "\n\n" +
            GUIDANCE + "\n\n" +
            "SENTENCES TO ANALYZE:\n" +
            numbered
        )

        # Responses API call with high reasoning effort
        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": "high"},
            instructions="You are a precise analyst. Follow the format rules exactly.",
            input=user_input,
        )

        out_text = response.output_text.strip()

        # Write raw JSONL lines; validate each line as JSON
        for line in out_text.splitlines():
            if line.strip():
                # Ensure valid JSON; if not, still write it
                try:
                    json.loads(line)
                except json.JSONDecodeError:
                    pass
                f.write(line.rstrip("\n") + "\n")

print(f"Wrote: {output_jsonl_path}")


Wrote: /Users/raspberry/Desktop/ECOLE wwi/wwi-arts.en.0_metaphor_analysis.jsonl
CPU times: user 59.8 ms, sys: 230 ms, total: 290 ms
Wall time: 12min 50s


In [33]:
df = pd.read_json('/Users/raspberry/Desktop/ECOLE wwi/wwi-arts.en.0_metaphor_analysis.jsonl', lines=True, encoding='utf-8')
df.to_csv('/Users/raspberry/Desktop/ECOLE wwi/wwi-arts.en.0_metaphor_analysis.csv', index=False, encoding='utf-8')